# EverFlavor AI - Stores, Restaurants and Products

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: planning sketch.** The sections, sources, output tables and functions below are the plan.
> The functions are stubs (`NotImplementedError`) and nothing here has been run or tested yet.
> "Run all" is safe: it only runs the setup and prints the plan's status.

---

## Purpose

The Sourcing Agent answers "where can I buy this?" and "where can I eat this?". This notebook collects:

- **Places:** grocery stores and restaurants near a location, with the cuisine they serve or the kind of
  products they sell (Google Places and OpenStreetMap).
- **Products by country:** packaged products, the brand, the country they come from and the countries
  and stores that sell them (Open Food Facts). This lets the agent suggest the brand a person knows from
  home, not just "soy sauce".
- **A link between the two:** recipe ingredient -> products -> stores that carry them.

**Safety rule (as in notebook 02):** anything about a specific store or restaurant (what it stocks, which
oil it fries in, whether a dish is halal) is **unverified** unless the place itself publishes it. The agent
says "check with the store" instead of stating it as a fact. Store data can add a caution, never clear a
restriction flag.

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | `data/interim/recipes_all.parquet` (ingredients, origin country) | Notebook 01 |
| Input | Google Places API (key in Colab Secrets or `config/.env`) | Section 2 |
| Input | OpenStreetMap through the Overpass API (no key) | Section 3 |
| Input | Open Food Facts product export (no key) | Section 4 |
| Output | `data/processed/places.parquet` (one row per place; Google rows keep only `place_id`, see section 2) | Sections 2-3 |
| Output | `data/processed/products_by_country.parquet` | Section 4 |
| Output | `data/processed/ingredient_products.parquet` | Section 5 |
| Output | `data/processed/region_coverage_stores.csv` | Section 6 |

## Contents

1. Setup
2. Google Places: stores and restaurants
3. OpenStreetMap: a free, open second source
4. Open Food Facts: products by country of origin
5. Linking recipe ingredients to products and stores
6. Coverage by country (which regions are ready)
7. Team decisions and next steps

---

## 1. Setup

The same setup as notebooks 01 and 02: it works unchanged in Google Colab, VS Code and Antigravity.
The Google Places key is read the same way as the USDA key: Colab Secrets in Colab, `config/.env` locally
(`GOOGLE_PLACES_API_KEY`). It is never printed and `config/.env` is never committed.

In [ ]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
try:
    import google.colab  # noqa: F401  # pyright: ignore[reportMissingImports]  (only exists in Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "checks", "environment", "charts", "sources", "parsing", "cuisine",
                      "ingredients", "flags", "diets", "nutrition", "features", "pipeline", "review",
                      "recommend", "reporting", "cooking"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
PILOT_AREA = {"name": "Houston, TX", "lat": 29.7604, "lng": -95.3698, "radius_m": 15000}  # team decision, section 7
PLACES_CACHE = Path("data/raw/places_cache")
OSM_CACHE = Path("data/raw/osm_cache")
OFF_EXPORT_DIR = Path("data/raw/off_export")
for folder in [PLACES_CACHE, OSM_CACHE, OFF_EXPORT_DIR, Path("data/processed")]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

---

## 2. Google Places: Stores and Restaurants

**API:** Places API (New), Text Search and Nearby Search. Queries such as "Ethiopian grocery store",
"Korean restaurant" or "halal butcher" inside the pilot area.

**Terms of service shape the design.** Google allows apps to store the **place ID** indefinitely, but not
most other content (names, ratings, opening hours) beyond short-term caching. So:

- the saved table keeps `place_id`, the query that found it and the date;
- the agent fetches the name, address and hours **live** when it shows a result;
- the field mask asks only for what is needed (cost is per request and depends on the fields).

**Cost control:** a fixed list of queries per pilot area, results cached by query, and `REFRESH_DOWNLOADS`
off by default, the same pattern as the Open Food Facts cache.

**Planned table** (`places.parquet`, Google rows):

| Column | Meaning |
|---|---|
| `place_id` | Google's ID (the only Google content stored) |
| `source` | `google` |
| `query` | The search that found it ("Ethiopian grocery store") |
| `category` | `grocery`, `restaurant`, `butcher`, `bakery` ... (from the query) |
| `cuisine_hint` | The cuisine in the query; a hint, not verified |
| `retrieved` | Date of the search |

In [ ]:
# Planned module: src/everflavor/stores.py
PLACE_QUERIES = {
    # category -> query templates; {cuisine} is filled from the pilot cuisines (section 7)
    "grocery": ["{cuisine} grocery store", "{cuisine} market"],
    "restaurant": ["{cuisine} restaurant"],
    "butcher": ["halal butcher", "kosher butcher"],
}


def search_places(query: str, area: dict, api_key: str, cache_dir: Path, refresh: bool = False) -> list[dict]:
    """Search Google Places (Text Search, New API) inside `area` and cache the place IDs.

    Planned: POST https://places.googleapis.com/v1/places:searchText with a field mask of
    places.id,places.types,places.primaryType; location bias from `area`; one cached JSON per query.
    Returns [{"place_id", "query", "types", "retrieved"}]. The key goes in the
    X-Goog-Api-Key header, never in the URL: requests puts the URL in its connection
    errors, so a key in the URL would be printed (see sources.usda_search).
    """
    raise NotImplementedError("sketch: Google Places search")


def place_details_live(place_id: str, api_key: str) -> dict:
    """Fetch name, address, hours and website for one place, at the moment the agent shows it (not stored)."""
    raise NotImplementedError("sketch: live place details")

---

## 3. OpenStreetMap: A Free, Open Second Source

**API:** Overpass API (no key). OpenStreetMap data is under the **ODbL** license: it can be stored and
used, with attribution ("© OpenStreetMap contributors"), and a database built from it must stay open.

Useful tags:

| Tag | Example |
|---|---|
| `amenity=restaurant` + `cuisine=*` | `cuisine=ethiopian`, `cuisine=korean;bbq` |
| `shop=supermarket`, `shop=convenience`, `shop=greengrocer` | Groceries |
| `shop=butcher` + `diet:halal=yes` / `diet:kosher=yes` | Halal or kosher butchers |
| `diet:vegan=*`, `diet:vegetarian=*` | Diet options at restaurants |

**Known weakness:** shops rarely say which country's products they sell (there is no standard tag for an
"Ethiopian grocery"). Section 5 adds a name-keyword hint ("Asian Market", "Mercado", "Halal") marked as a
guess, and Google Places fills the gap. **Usage policy:** few, cached queries (the public Overpass servers
ask for fair use).

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSM_TAGS = {
    "restaurant": '["amenity"="restaurant"]',
    "grocery": '["shop"~"supermarket|convenience|greengrocer|deli"]',
    "butcher": '["shop"="butcher"]',
}


def osm_places(area: dict, category: str, cache_dir: Path, refresh: bool = False) -> pd.DataFrame:
    """Download OpenStreetMap places of one category around `area` (Overpass `around:` query), cached.

    Returns one row per place: osm_id, name, lat, lng, category, cuisine (split on ";"),
    diet_halal, diet_kosher, diet_vegan, opening_hours, website, source="osm".
    """
    raise NotImplementedError("sketch: Overpass query")

---

## 4. Open Food Facts: Products by Country of Origin

The compound-ingredient check in notebook 01 (section 5.13) showed that the **search API** is slow
(about 10 searches a minute) and often busy (HTTP 503). For thousands of products we use the **full
export** instead: one download, no rate limit.

**Source (checked 2026-10-04):** the Parquet export on Hugging Face, `openfoodfacts/product-database`,
file `food.parquet` (about 4.2 GB, about 4.8 million products, ODbL). Parquet lets us read only the
columns below, so it fits in Colab's memory even though the file is large; the download itself goes to
`data/raw/off_export/` (never committed).

Fields used:

| Field | Meaning |
|---|---|
| `product_name`, `brands`, `categories_tags` | What the product is |
| `origins_tags` | Where the ingredients come from |
| `manufacturing_places_tags` | Where it is made |
| `countries_tags` | Countries where it is **sold** |
| `stores_tags` | Store chains that sell it (filled in by volunteers, incomplete) |
| `allergens_tags`, `traces_tags`, `ingredients_analysis_tags` | Allergens, "may contain", vegan / vegetarian / palm oil status |

**Planned table** (`products_by_country.parquet`): one row per product with the fields above, plus
`home_country` (origin, or the manufacturing country when origin is empty) and `sold_in_us` (true when
`countries_tags` has `en:united-states`).

In [ ]:
OFF_FIELDS = ["code", "product_name", "brands", "categories_tags", "origins_tags",
              "manufacturing_places_tags", "countries_tags", "stores_tags",
              "allergens_tags", "traces_tags", "ingredients_analysis_tags"]


def off_products_by_country(export_dir: Path, refresh: bool = False) -> pd.DataFrame:
    """Read the Open Food Facts export (only OFF_FIELDS, in chunks) and add home_country and sold_in_us.

    Source: https://huggingface.co/datasets/openfoodfacts/product-database (food.parquet, ~4.2 GB, ODbL).
    Read with pyarrow, columns=OFF_FIELDS, so the whole file is never loaded into memory.
    """
    raise NotImplementedError("sketch: Open Food Facts export")

---

## 5. Linking Recipe Ingredients to Products and Stores

1. **Ingredient -> products:** match normalized ingredient names (notebook 01) to product categories and
   names, the same name matching as the compound-ingredient check.
2. **Products -> home country:** "this Sichuan recipe needs doubanjiang; these brands come from China
   and are sold in the US".
3. **Products -> stores:** from `stores_tags` where it exists; otherwise only "try an Asian grocery near
   you" from the places in sections 2-3. Never "store X has it".

**Planned table** (`ingredient_products.parquet`): `ingredient`, `code`, `brand`, `home_country`,
`sold_in_us`, `match` (`exact` / `category`), `confidence`.

In [ ]:
def link_ingredients_to_products(ingredients: pd.Series, products: pd.DataFrame) -> pd.DataFrame:
    """Match normalized ingredient names to Open Food Facts products (name and category match)."""
    raise NotImplementedError("sketch: ingredient -> product matching")


def suggest_where_to_buy(ingredient: str, origin_country: str, links: pd.DataFrame, places: pd.DataFrame) -> dict:
    """Return brands from `origin_country` and nearby store types, every store claim marked unverified."""
    raise NotImplementedError("sketch: sourcing suggestion")

---

## 6. Coverage by Country

For each origin country in the recipes: how many recipes, how many of their ingredients link to a product,
how many products come from that country and are sold in the US, and how many nearby places serve that
cuisine. This says which regions are ready for the Sourcing Agent and where data is thin.

In [ ]:
def region_coverage(recipes: pd.DataFrame, links: pd.DataFrame, products: pd.DataFrame,
                    places: pd.DataFrame) -> pd.DataFrame:
    """One row per origin country: recipes, ingredients_linked_pct, home_products_sold_in_us, nearby_places."""
    raise NotImplementedError("sketch: coverage report")

---

## 7. Team Decisions and Next Steps

**Decisions needed before building:**

1. **Pilot area:** the city and radius for store searches (the setup cell has Houston as a placeholder).
2. **Pilot cuisines:** which countries first. By recipe count in notebook 01: United States 131k, Italy 25k,
   Mexico 15k, France 8.7k, India 7.5k, China 6.0k, United Kingdom 4.7k, Canada 4.6k, Greece 4.4k,
   Australia 2.8k, Thailand 2.5k, Japan 2.3k, Spain 1.9k, Morocco 1.6k.
   Countries with fewer recipes are where sourcing help matters most, so the pilot could mix both.
3. **Google Places budget:** a monthly cap for API calls.

**Build order:** OpenStreetMap first (free, no key), then Open Food Facts export, then Google Places.

---

## Plan status

What is built, what is a stub, and what needs a team decision first.

In [ ]:
PLAN = pd.DataFrame([
    ("Plan, sources, output tables", "written"),
    ("2 Google Places search", "stub - needs pilot area and budget"),
    ("3 OpenStreetMap places", "stub - build first (free)"),
    ("4 Open Food Facts export", "stub - source checked (food.parquet, 4.2 GB)"),
    ("5 Ingredient -> product -> store", "stub"),
    ("6 Coverage by country", "stub"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))